# 🏙️ Ontario Places-to-Live RAG Assistant

A grounded, **retrieval-augmented generation** system that answers questions about
living in Ontario municipalities — population, density, amenities, and public
description — by **retrieving from a knowledge base built entirely from real,
public data** and generating **cited, source-grounded** answers with a small open
LLM that runs **free in Google Colab**.

> **What this is:** a descriptive, sourced research assistant.
> **What this is *not*:** a tool that tells you where to move. It reports what the
> data says, cites where each fact came from, and says *"I don't have data on that"*
> when the knowledge base is silent — rather than guessing.

---

### Architecture

```
                         ┌───────────────────────────────────────────────┐
   REAL PUBLIC SOURCES   │  INGESTION (cached, rate-limited, restart-safe) │
   ┌──────────────────┐  │                                                 │
   │ Wikipedia REST + │──┤  per municipality:                              │
   │   Action API     │  │    • Wikipedia extract  (descriptive text)      │
   ├──────────────────┤  │    • OSM Overpass       (amenity counts)        │
   │ OSM Overpass API │──┤    • curated 2021 Census backbone (facts)       │
   ├──────────────────┤  │    • StatCan WDS        (optional enrichment)    │
   │ StatCan WDS (opt)│  └───────────────────────┬─────────────────────────┘
   ├──────────────────┤                          │  raw corpus (JSON on disk)
   │ 2021 Census      │                          ▼
   │  backbone (facts)│      ┌───────────────────────────────────────┐
   └──────────────────┘      │ PROCESSING: clean → fact-sentences →   │
                             │ chunk (token-budget, self-contained)   │
                             └───────────────────┬───────────────────┘
                                                 │ chunks + metadata
                                                 ▼
        ┌───────────────┐   embed    ┌───────────────────────────┐
        │ BGE-small-v1.5│──────────▶ │ FAISS IndexFlatIP (cosine) │
        │  (sentence-t) │            │  + parallel chunk metadata │
        └───────────────┘            └─────────────┬─────────────┘
                                                    │ top-k (+ metadata filter)
                    ┌───────────────────────────────▼───────────────────┐
   question ──────▶ │ RETRIEVER → grounding prompt → Qwen2.5-3B-Instruct │──▶ cited answer
                    └────────────────────────────────────────────────────┘
                                                    │
                                                    ▼
                                   EVALUATION: recall@k · faithfulness ·
                                   numeric-support · abstention behaviour
```

### Honest limitations (read before trusting an answer)
- **Retrieval can miss.** If the relevant chunk isn't in the top-k, the model won't see it.
- **Small local models are weaker synthesizers.** A 3B model can still misread context; it is not GPT-4.
- **OSM coverage is uneven.** Amenity counts use a bounding-box approximation and depend on how well an area is mapped — treat them as *rough relative* signals, not exact inventories.
- **Census/StatCan coverage is uneven** and boundaries (city vs. census subdivision vs. metro area) don't always align across sources.
- **No sentiment by default.** Sentiment is only included if you supply a cited CSV; nothing is fabricated.

Every section below narrates **what** it does, **why**, **why this method**, **computational considerations**, and **limitations**.

## 1 · Environment

**What:** install dependencies, detect GPU, set seeds, build the project directory
tree, configure logging, and freeze a single `CONFIG` object that every later cell
reads from.

**Why this way:** a single frozen config makes the whole notebook reproducible and
restart-safe — there are no magic constants scattered across cells. We *don't*
hard-pin versions aggressively (Colab ships recent builds); instead, later cells
**introspect installed library signatures** and pass only supported kwargs, which
is far more robust than betting on one exact version.

**Computational considerations:** everything here is free-tier friendly. The
heaviest object we load later (a 3B-parameter LLM in fp16) needs ~6 GB of VRAM and
fits a free Colab T4 (16 GB). On CPU-only runtimes it still runs, just slowly.

In [1]:
#@title Install dependencies (quiet, resilient)
import subprocess, sys, importlib

def _pip_install(pkgs):
    """Install packages, tolerating Colab's pre-installed versions."""
    cmd = [sys.executable, "-m", "pip", "install", "-q", *pkgs]
    try:
        subprocess.run(cmd, check=True)
    except subprocess.CalledProcessError as e:
        print(f"[warn] pip install returned non-zero for {pkgs}: {e}. "
              f"Continuing — a compatible version may already be present.")

# Core RAG stack. faiss-cpu is used because free Colab GPUs are memory-constrained
# and a flat index over a few thousand vectors is instant on CPU anyway.
_REQUIRED = [
    "sentence-transformers>=2.7.0",
    "faiss-cpu>=1.7.4",
    "transformers>=4.44.0",
    "accelerate>=0.33.0",
    "requests>=2.31.0",
    "tqdm>=4.66.0",
    "pandas>=2.0.0",
]
_pip_install(_REQUIRED)
print("Dependency install step complete.")

Dependency install step complete.


In [2]:
#@title Imports, GPU detection, seeds, logging
import os, sys, json, time, math, hashlib, logging, random, re, html, unicodedata, inspect
from dataclasses import dataclass, field, asdict
from typing import Any, Callable, Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import requests
from tqdm.auto import tqdm

# ---- logging: one place, timestamped, INFO by default --------------------- #
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-7s | %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("ontario-rag")

# ---- reproducibility ------------------------------------------------------ #
SEED = 42
random.seed(SEED); np.random.seed(SEED)
try:
    import torch
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
    _HAS_TORCH = True
except Exception as e:  # torch should exist in Colab, but degrade cleanly
    torch = None
    _HAS_TORCH = False
    log.warning("PyTorch unavailable (%s); generation will be limited.", e)

# ---- device detection ----------------------------------------------------- #
def detect_device() -> str:
    if _HAS_TORCH and torch.cuda.is_available():
        name = torch.cuda.get_device_name(0)
        vram = torch.cuda.get_device_properties(0).total_memory / 1e9
        log.info("GPU detected: %s (%.1f GB VRAM)", name, vram)
        return "cuda"
    log.warning("No GPU detected — running on CPU. Embedding and generation will "
                "be slower; consider Runtime > Change runtime type > GPU.")
    return "cpu"

DEVICE = detect_device()

In [3]:
#@title Project directories + frozen CONFIG
from pathlib import Path

@dataclass(frozen=True)
class Config:
    """Single source of truth. Frozen so no cell can mutate it by accident."""
    # --- paths (all relative; no hardcoded absolute paths) ----------------- #
    root: str = "ontario_rag_data"
    raw_dir: str = "ontario_rag_data/raw"          # cached API responses
    http_cache_dir: str = "ontario_rag_data/http_cache"
    artifacts_dir: str = "ontario_rag_data/artifacts"  # index, chunks, reports

    # --- models (verified real + ungated as of build) ---------------------- #
    embed_model: str = "BAAI/bge-small-en-v1.5"
    embed_model_fallback: str = "sentence-transformers/all-MiniLM-L6-v2"
    # BGE-v1.5 retrieval instruction for QUERIES only (not passages):
    bge_query_instruction: str = "Represent this sentence for searching relevant passages: "
    gen_models: Tuple[str, ...] = (
        "Qwen/Qwen2.5-3B-Instruct",      # primary: ungated, ~6GB fp16, fits T4
        "microsoft/Phi-4-mini-instruct", # fallback 1: ungated
        "Qwen/Qwen2.5-1.5B-Instruct",    # fallback 2: smaller, CPU-friendlier
    )

    # --- chunking / retrieval ---------------------------------------------- #
    chunk_max_tokens: int = 220
    chunk_overlap_sentences: int = 1
    top_k: int = 5
    embed_batch_size: int = 64
    gen_max_new_tokens: int = 400
    # CPU-friendly fallbacks, auto-used when no GPU is detected (see below):
    gen_models_cpu: Tuple[str, ...] = (
        "Qwen/Qwen2.5-1.5B-Instruct",
        "Qwen/Qwen2.5-0.5B-Instruct",
    )
    gen_max_new_tokens_cpu: int = 200
    eval_max_questions_cpu: int = 12

    # --- ingestion knobs --------------------------------------------------- #
    wiki_user_agent: str = ("OntarioPlacesToLiveRAG/1.0 "
                            "(portfolio project; contact: user@example.com)")
    overpass_endpoints: Tuple[str, ...] = (
        "https://overpass-api.de/api/interpreter",
        "https://overpass.private.coffee/api/interpreter",  # mirror
        "https://overpass.kumi.systems/api/interpreter",     # mirror (often busy)
    )
    overpass_bbox_deg: float = 0.12          # ~13 km half-width bounding box
    request_timeout_s: int = 60
    max_retries: int = 4
    backoff_base_s: float = 2.0

    # StatCan WDS is REAL and wired below, but OFF by default so a fresh Colab
    # run is guaranteed clean: full census tables are large and slow to match by
    # municipality. Populations come from the cited 2021-Census backbone + the
    # Wikipedia demographic text. Flip this to True to enrich live (see §4c).
    use_statcan_wds: bool = False

    seed: int = 42

CONFIG = Config()

for _d in (CONFIG.root, CONFIG.raw_dir, CONFIG.http_cache_dir, CONFIG.artifacts_dir):
    Path(_d).mkdir(parents=True, exist_ok=True)

log.info("Config frozen. Embed=%s  Gen=%s", CONFIG.embed_model, CONFIG.gen_models[0])
log.info("Artifacts will be written to ./%s", CONFIG.artifacts_dir)

## 2 · Municipality backbone (real public facts)

**What:** a curated table of real Ontario municipalities with their **2021 Census
of Population** counts, **land area** (where confidently known), census division,
and coordinates.

**Why:** these are stable public facts (not URLs or fabricated numbers). They give
every municipality a reliable spine of structured facts *and* the coordinates the
OpenStreetMap step needs. Population is cited to *Statistics Canada, 2021 Census of
Population*; the Wikipedia step independently pulls demographic prose, giving a
soft cross-check.

**Limitations:** "population" here is the **city proper (census subdivision)**, not
the metropolitan area — Toronto's CSD (~2.79 M) is much smaller than the Toronto
CMA (~6 M). Where we weren't confident of an exact land area, it is left blank and
no density is emitted (we never invent a number to fill a gap).

In [4]:
#@title Curated 2021-Census backbone (source: Statistics Canada, 2021 Census)
# pop_2021 = 2021 Census of Population, census subdivision (city proper).
# land_area_km2 left as None where not confidently known -> density simply omitted.
MUNICIPALITIES: List[Dict[str, Any]] = [
    {"name": "Toronto",        "wiki": "Toronto",              "pop_2021": 2794356, "land_area_km2": 630.2,  "cd": "Toronto (single-tier)",        "lat": 43.7417, "lon": -79.3733},
    {"name": "Ottawa",         "wiki": "Ottawa",               "pop_2021": 1017449, "land_area_km2": 2790.2, "cd": "Ottawa (single-tier)",         "lat": 45.4215, "lon": -75.6972},
    {"name": "Mississauga",    "wiki": "Mississauga",          "pop_2021": 717961,  "land_area_km2": 292.4,  "cd": "Peel Region",                  "lat": 43.5890, "lon": -79.6441},
    {"name": "Brampton",       "wiki": "Brampton",             "pop_2021": 656480,  "land_area_km2": 265.9,  "cd": "Peel Region",                  "lat": 43.7315, "lon": -79.7624},
    {"name": "Hamilton",       "wiki": "Hamilton, Ontario",    "pop_2021": 569353,  "land_area_km2": 1118.3, "cd": "Hamilton (single-tier)",       "lat": 43.2557, "lon": -79.8711},
    {"name": "London",         "wiki": "London, Ontario",      "pop_2021": 422324,  "land_area_km2": 420.6,  "cd": "Middlesex County",             "lat": 42.9849, "lon": -81.2453},
    {"name": "Markham",        "wiki": "Markham, Ontario",     "pop_2021": 338503,  "land_area_km2": 210.6,  "cd": "York Region",                  "lat": 43.8561, "lon": -79.3370},
    {"name": "Vaughan",        "wiki": "Vaughan",              "pop_2021": 323103,  "land_area_km2": 273.5,  "cd": "York Region",                  "lat": 43.8361, "lon": -79.4983},
    {"name": "Kitchener",      "wiki": "Kitchener, Ontario",   "pop_2021": 256885,  "land_area_km2": 136.9,  "cd": "Waterloo Region",              "lat": 43.4516, "lon": -80.4925},
    {"name": "Windsor",        "wiki": "Windsor, Ontario",     "pop_2021": 229660,  "land_area_km2": 146.3,  "cd": "Essex County",                 "lat": 42.3149, "lon": -83.0364},
    {"name": "Oakville",       "wiki": "Oakville, Ontario",    "pop_2021": 213759,  "land_area_km2": 138.9,  "cd": "Halton Region",                "lat": 43.4675, "lon": -79.6877},
    {"name": "Richmond Hill",  "wiki": "Richmond Hill, Ontario","pop_2021": 202022, "land_area_km2": 100.9,  "cd": "York Region",                  "lat": 43.8828, "lon": -79.4403},
    {"name": "Burlington",     "wiki": "Burlington, Ontario",  "pop_2021": 186948,  "land_area_km2": 185.7,  "cd": "Halton Region",                "lat": 43.3255, "lon": -79.7990},
    {"name": "Oshawa",         "wiki": "Oshawa",               "pop_2021": 175383,  "land_area_km2": 145.7,  "cd": "Durham Region",                "lat": 43.8971, "lon": -78.8658},
    {"name": "Greater Sudbury","wiki": "Greater Sudbury",      "pop_2021": 166004,  "land_area_km2": 3201.0, "cd": "Greater Sudbury (single-tier)","lat": 46.4917, "lon": -80.9930},
    {"name": "Barrie",         "wiki": "Barrie",               "pop_2021": 147829,  "land_area_km2": 99.0,   "cd": "Simcoe County",                "lat": 44.3894, "lon": -79.6903},
    {"name": "Guelph",         "wiki": "Guelph",               "pop_2021": 143740,  "land_area_km2": 87.2,   "cd": "Wellington County",            "lat": 43.5448, "lon": -80.2482},
    {"name": "Whitby",         "wiki": "Whitby, Ontario",      "pop_2021": 138501,  "land_area_km2": 146.5,  "cd": "Durham Region",                "lat": 43.8975, "lon": -78.9428},
    {"name": "Cambridge",      "wiki": "Cambridge, Ontario",   "pop_2021": 138479,  "land_area_km2": 112.8,  "cd": "Waterloo Region",              "lat": 43.3616, "lon": -80.3144},
    {"name": "St. Catharines", "wiki": "St. Catharines",       "pop_2021": 136803,  "land_area_km2": 96.1,   "cd": "Niagara Region",               "lat": 43.1594, "lon": -79.2469},
    {"name": "Kingston",       "wiki": "Kingston, Ontario",    "pop_2021": 132485,  "land_area_km2": 451.2,  "cd": "Frontenac County",             "lat": 44.2312, "lon": -76.4860},
    {"name": "Waterloo",       "wiki": "Waterloo, Ontario",    "pop_2021": 121436,  "land_area_km2": 64.1,   "cd": "Waterloo Region",              "lat": 43.4643, "lon": -80.5204},
    {"name": "Thunder Bay",    "wiki": "Thunder Bay",          "pop_2021": 108843,  "land_area_km2": 328.2,  "cd": "Thunder Bay District",         "lat": 48.3809, "lon": -89.2477},
    {"name": "Peterborough",   "wiki": "Peterborough, Ontario","pop_2021": 83651,   "land_area_km2": 64.1,   "cd": "Peterborough County",          "lat": 44.3091, "lon": -78.3197},
]

# Quick-run knob: set to a small int (e.g. 6) to ingest fewer cities while testing.
# 0 = use all. Fewer municipalities => fewer API calls => faster first run.
MAX_MUNICIPALITIES = 0
if MAX_MUNICIPALITIES:
    MUNICIPALITIES = MUNICIPALITIES[:MAX_MUNICIPALITIES]

MUNI_NAMES = [m["name"] for m in MUNICIPALITIES]

# Validate the backbone (fail loudly on malformed data, never silently proceed).
_seen = set()
for m in MUNICIPALITIES:
    assert m["name"] and m["name"] not in _seen, f"dup/empty name: {m['name']}"
    _seen.add(m["name"])
    assert -90 <= m["lat"] <= 90 and -180 <= m["lon"] <= 0, f"bad coords: {m['name']}"
    assert isinstance(m["pop_2021"], int) and m["pop_2021"] > 0, f"bad pop: {m['name']}"

_backbone_df = pd.DataFrame(MUNICIPALITIES)
log.info("Backbone loaded: %d municipalities (2021 Census).", len(MUNICIPALITIES))
_backbone_df[["name", "pop_2021", "land_area_km2", "cd"]].head(10)

,name,pop_2021,land_area_km2,cd
0,Toronto,2794356,630.2,Toronto (single-tier)
1,Ottawa,1017449,2790.2,Ottawa (single-tier)
2,Mississauga,717961,292.4,Peel Region
3,Brampton,656480,265.9,Peel Region
4,Hamilton,569353,1118.3,Hamilton (single-tier)
5,London,422324,420.6,Middlesex County
6,Markham,338503,210.6,York Region
7,Vaughan,323103,273.5,York Region
8,Kitchener,256885,136.9,Waterloo Region
9,Windsor,229660,146.3,Essex County


## 3 · HTTP utilities — caching, rate limiting, backoff

**What:** one shared HTTP layer used by every ingestion source. It (a) caches every
response to disk keyed by a hash of the request, so re-runs and restarts never
re-download; (b) enforces a **minimum interval per host** so we never hammer a
public API; and (c) **retries with exponential backoff**, honouring `Retry-After`
on HTTP 429.

**Why this matters:** Wikipedia, Overpass, and StatCan are free community/public
resources. Respecting their rate limits isn't optional — it's the difference
between a good citizen and a blocked IP. Caching also makes the notebook *fast* on
the second run and **restart-safe** (a Colab disconnect loses nothing already
fetched).

In [5]:
#@title Disk-cached, rate-limited HTTP with exponential backoff
from urllib.parse import urlparse

class RateLimiter:
    """Enforce a minimum wall-clock interval between calls to the same host."""
    def __init__(self, min_interval_s: float):
        self.min_interval = min_interval_s
        self._last: Dict[str, float] = {}
    def wait(self, url: str) -> None:
        host = urlparse(url).netloc
        now = time.monotonic()
        prev = self._last.get(host)
        if prev is not None:
            delta = now - prev
            if delta < self.min_interval:
                time.sleep(self.min_interval - delta)
        self._last[host] = time.monotonic()

# Conservative pacing per host (seconds between requests).
_RATE = RateLimiter(min_interval_s=1.0)

# Public APIs (notably overpass-api.de since 2026) reject requests lacking a
# descriptive User-Agent / Accept header with HTTP 406. Always send both.
_DEFAULT_HEADERS = {"User-Agent": CONFIG.wiki_user_agent,
                    "Accept": "application/json, */*"}

def _cache_path(key: str) -> Path:
    h = hashlib.sha256(key.encode("utf-8")).hexdigest()[:24]
    return Path(CONFIG.http_cache_dir) / f"{h}.json"

def cached_request(method: str, url: str, *, params=None, data=None,
                   headers=None, expect: str = "json", cache_key: str = None,
                   timeout: int = None, max_retries: int = None) -> Any:
    """GET/POST with disk cache, pacing, and backoff. Returns json|text.

    Raises requests.RequestException only after exhausting retries, so callers
    can decide whether a given source failure is fatal or merely degrades scope.
    """
    timeout = timeout or CONFIG.request_timeout_s
    max_retries = max_retries or CONFIG.max_retries
    key = cache_key or f"{method}|{url}|{json.dumps(params, sort_keys=True)}|{data}"
    cpath = _cache_path(key)
    if cpath.exists():
        try:
            payload = json.loads(cpath.read_text(encoding="utf-8"))
            return payload["value"]
        except Exception:
            cpath.unlink(missing_ok=True)  # corrupt cache -> refetch

    merged_headers = dict(_DEFAULT_HEADERS)
    if headers:
        merged_headers.update(headers)
    last_err = None
    for attempt in range(max_retries):
        _RATE.wait(url)
        try:
            resp = requests.request(method, url, params=params, data=data,
                                    headers=merged_headers, timeout=timeout)
            if resp.status_code == 429:
                retry_after = float(resp.headers.get("Retry-After", 0) or 0)
                # Honour Retry-After but cap it so an overloaded mirror can't stall the run.
                wait_s = min(max(retry_after, CONFIG.backoff_base_s * (2 ** attempt)), 20.0)
                log.warning("429 from %s; backing off %.0fs (attempt %d/%d)",
                            urlparse(url).netloc, wait_s, attempt + 1, max_retries)
                time.sleep(wait_s)
                continue
            resp.raise_for_status()
            value = resp.json() if expect == "json" else resp.text
            cpath.write_text(json.dumps({"value": value}, ensure_ascii=False),
                             encoding="utf-8")
            return value
        except requests.HTTPError as e:
            status = getattr(e.response, "status_code", None)
            # Deterministic client errors (e.g. 406) won't change on retry -> fail
            # fast so the caller can move to the next endpoint.
            if status and 400 <= status < 500 and status != 429:
                raise
            last_err = e
            wait_s = CONFIG.backoff_base_s * (2 ** attempt)
            log.warning("HTTP %s on %s; retry in %.0fs (attempt %d/%d)",
                        status, urlparse(url).netloc, wait_s, attempt + 1, max_retries)
            time.sleep(wait_s)
        except (requests.RequestException, ValueError) as e:
            last_err = e
            wait_s = CONFIG.backoff_base_s * (2 ** attempt)
            log.warning("Request error (%s) on %s; retry in %.0fs (attempt %d/%d)",
                        type(e).__name__, urlparse(url).netloc, wait_s,
                        attempt + 1, max_retries)
            time.sleep(wait_s)
    raise requests.RequestException(
        f"Exhausted {max_retries} retries for {url}: {last_err}")

log.info("HTTP layer ready (cache=./%s).", CONFIG.http_cache_dir)

## 4 · Ingestion

For each municipality we pull from **real, no-auth public sources**:

- **4a · Wikipedia** (Action API `prop=extracts`) — descriptive prose: history,
  economy, geography, demographics. Rich, well-written context.
- **4b · OpenStreetMap Overpass** — *counts* of real mapped amenities (parks,
  schools, hospitals, libraries, pharmacies, transit stops, supermarkets,
  restaurants) inside a bounding box around the city centre.
- **4c · StatCan Web Data Service** — real census tables via `getFullTableDownloadCSV`.
  **Wired and real, but off by default** (`CONFIG.use_statcan_wds`) so a fresh run
  is guaranteed clean; population already comes from the cited 2021-Census backbone.
- **4e · Wikivoyage** (same MediaWiki API) — editorial, qualitative "what's it like"
  prose (e.g. *trendy*, *gentrified*, *quiet residential*). Travel-writer tone,
  **not** resident reviews or measured sentiment — labelled as such so answers stay honest.

Everything is cached; partial per-source failures **degrade gracefully** (logged,
skipped) but if *nothing at all* is retrieved across every source we **fail loudly**
rather than fabricate.

In [6]:
#@title 4a · Wikipedia extracts (Action API, redirect-following)
WIKI_API = "https://en.wikipedia.org/w/api.php"

def fetch_wikipedia(title: str) -> Optional[Dict[str, Any]]:
    """Return {title, url, text} plain-text extract for a page, or None."""
    params = {
        "action": "query", "prop": "extracts", "explaintext": 1,
        "redirects": 1, "format": "json", "titles": title,
        "formatversion": 2,
    }
    headers = {"User-Agent": CONFIG.wiki_user_agent}
    try:
        data = cached_request("GET", WIKI_API, params=params, headers=headers,
                              cache_key=f"wiki|{title}")
    except requests.RequestException as e:
        log.warning("Wikipedia fetch failed for %r: %s", title, e)
        return None
    pages = data.get("query", {}).get("pages", [])
    if not pages:
        return None
    page = pages[0]
    text = page.get("extract", "") or ""
    if not text.strip():
        log.warning("Wikipedia returned empty extract for %r", title)
        return None
    real_title = page.get("title", title)
    url = "https://en.wikipedia.org/wiki/" + real_title.replace(" ", "_")
    return {"title": real_title, "url": url, "text": text}

In [7]:
#@title 4e · Wikivoyage descriptive text (editorial "vibe", not resident reviews)
WIKIVOYAGE_API = "https://en.wikivoyage.org/w/api.php"

def fetch_wikivoyage(name: str) -> Optional[Dict[str, Any]]:
    """Return {title,url,text} from Wikivoyage for an Ontario city, or None.

    Wikivoyage runs the same TextExtracts API as Wikipedia; its articles open with
    a free-form 'Understand' description in travel-writer voice (e.g. 'trendy',
    'gentrified', 'quiet residential'). This is a QUALITATIVE signal about what
    areas are like -- NOT resident reviews or measured sentiment. Chunks are tagged
    source='wikivoyage' so answers can be honest about the tone.

    Ontario cities are disambiguated on Wikivoyage as 'Name (Ontario)'; we try that
    first (correct for London/Hamilton/Windsor/etc.), then the bare name, following
    redirects. Uses GET (POST has a known explaintext quirk).
    """
    headers = {"User-Agent": CONFIG.wiki_user_agent}
    for title in (f"{name} (Ontario)", name):
        params = {"action": "query", "prop": "extracts", "explaintext": 1,
                  "redirects": 1, "format": "json", "formatversion": 2,
                  "titles": title}
        try:
            data = cached_request("GET", WIKIVOYAGE_API, params=params,
                                  headers=headers, cache_key=f"wikivoyage|{title}")
        except requests.RequestException as e:
            log.warning("Wikivoyage fetch failed for %r: %s", title, e)
            continue
        pages = data.get("query", {}).get("pages", [])
        if not pages:
            continue
        page = pages[0]
        if page.get("missing"):
            continue
        text = (page.get("extract") or "").strip()
        if len(text) < 200:          # skip stubs / near-empty guides
            continue
        real = page.get("title", title)
        url = "https://en.wikivoyage.org/wiki/" + real.replace(" ", "_")
        return {"title": real, "url": url, "text": text}
    return None

In [8]:
#@title 4b · OpenStreetMap Overpass amenity counts
# category -> list of (osm_key, osm_value) selectors unioned together.
OSM_CATEGORIES: Dict[str, List[Tuple[str, str]]] = {
    "park":         [("leisure", "park")],
    "school":       [("amenity", "school")],
    "hospital":     [("amenity", "hospital")],
    "pharmacy":     [("amenity", "pharmacy")],
    "library":      [("amenity", "library")],
    "transit_stop": [("highway", "bus_stop"), ("railway", "station")],
    "supermarket":  [("shop", "supermarket")],
    "restaurant":   [("amenity", "restaurant")],
}

def _bbox_for(muni: Dict[str, Any]) -> Tuple[float, float, float, float]:
    d = CONFIG.overpass_bbox_deg
    s, w = muni["lat"] - d, muni["lon"] - d
    n, e = muni["lat"] + d, muni["lon"] + d
    return (s, w, n, e)

def build_overpass_query(muni: Dict[str, Any]) -> str:
    """One query emitting an ordered `out count` per category (OSM_CATEGORIES order)."""
    s, w, n, e = _bbox_for(muni)
    bbox = f"({s:.5f},{w:.5f},{n:.5f},{e:.5f})"
    lines = ["[out:json][timeout:90];"]
    for cat, selectors in OSM_CATEGORIES.items():
        members = []
        for k, v in selectors:
            members.append(f'  node["{k}"="{v}"]{bbox};')
            members.append(f'  way["{k}"="{v}"]{bbox};')
        lines.append("(")
        lines.extend(members)
        lines.append(f") -> .{cat};")
        lines.append(f".{cat} out count;")
    return "\n".join(lines)

def parse_overpass_counts(data: Dict[str, Any]) -> Dict[str, int]:
    """Map ordered `count` elements back to category names by position."""
    counts = [el for el in data.get("elements", []) if el.get("type") == "count"]
    cats = list(OSM_CATEGORIES.keys())
    out: Dict[str, int] = {}
    for cat, el in zip(cats, counts):
        total = el.get("tags", {}).get("total")
        if total is not None:
            out[cat] = int(total)
    return out

# Module-level so the optional health-check cell (§4c-bis) can update these.
OSM_ENABLED = True
OVERPASS_ENDPOINTS = list(CONFIG.overpass_endpoints)

def fetch_overpass(muni: Dict[str, Any]) -> Dict[str, int]:
    """Amenity counts for one municipality; tries mirrors; {} on total failure.

    Returns {} immediately when OSM_ENABLED is False (graceful skip-OSM mode).
    """
    if not OSM_ENABLED:
        return {}
    query = build_overpass_query(muni)
    for endpoint in OVERPASS_ENDPOINTS:
        try:
            data = cached_request("POST", endpoint,
                                  data={"data": query}, expect="json",
                                  cache_key=f"overpass|{muni['name']}|{query}",
                                  timeout=45, max_retries=1)
            counts = parse_overpass_counts(data)
            if counts:
                return counts
        except requests.RequestException as e:
            log.warning("Overpass endpoint %s failed for %s: %s",
                        urlparse(endpoint).netloc, muni["name"], e)
            continue
    log.warning("All Overpass endpoints failed for %s; skipping amenities.", muni["name"])
    return {}

In [9]:
#@title 4c · StatCan Web Data Service client (real; off by default)
import io, zipfile
STATCAN_WDS = "https://www150.statcan.gc.ca/t1/wds/rest/getFullTableDownloadCSV/{pid}/{lang}"

def statcan_full_table(product_id: str, lang: str = "en") -> Optional[pd.DataFrame]:
    """Download a full StatCan table via WDS (real endpoint) -> DataFrame or None.

    Two hops: (1) WDS returns a JSON pointer to a zip; (2) we fetch + unzip the CSV.
    Off by default because full census tables are large and matching rows to our
    municipalities reliably is fragile — see the limitations section.
    """
    pid = product_id.replace("-", "")[:8]
    url = STATCAN_WDS.format(pid=pid, lang=lang)
    try:
        meta = cached_request("GET", url, cache_key=f"statcan_meta|{pid}|{lang}")
        if meta.get("status") != "SUCCESS":
            log.warning("StatCan WDS non-success for %s: %s", pid, meta.get("status"))
            return None
        zip_url = meta["object"]
        _RATE.wait(zip_url)
        resp = requests.get(zip_url, timeout=180)
        resp.raise_for_status()
        with zipfile.ZipFile(io.BytesIO(resp.content)) as zf:
            csvs = [x for x in zf.namelist() if x.endswith(".csv") and "MetaData" not in x]
            if not csvs:
                return None
            with zf.open(csvs[0]) as f:
                return pd.read_csv(f, low_memory=False)
    except (requests.RequestException, zipfile.BadZipFile, KeyError) as e:
        log.warning("StatCan WDS failed for %s: %s", product_id, e)
        return None

### 4c-bis · Overpass health check + skip toggle (optional but recommended)

Public Overpass mirrors go up and down, and since 2026 `overpass-api.de` returns
**406** for requests without proper headers. This cell probes each mirror with a
tiny query, keeps only the ones that currently answer, and — if none respond —
**cleanly disables OSM** so the run still completes from Wikipedia + census (amenity
questions then correctly abstain). Set `FORCE_SKIP_OSM = True` to skip OSM outright.
Re-run it later to re-enable OSM once a mirror frees up.

In [10]:
#@title 4c-bis · Probe Overpass mirrors / set skip toggle
FORCE_SKIP_OSM = False  #@param {type:"boolean"}

def overpass_health_check(verbose: bool = True) -> None:
    """Probe mirrors; keep working ones in OVERPASS_ENDPOINTS; toggle OSM_ENABLED."""
    global OVERPASS_ENDPOINTS, OSM_ENABLED
    if FORCE_SKIP_OSM:
        OSM_ENABLED = False
        print("FORCE_SKIP_OSM is on -> OSM amenity counts disabled for this run.")
        return
    # Probe with a REAL full query (lightest city) so the result predicts what the
    # actual per-city amenity queries will do -- a trivial probe can pass while the
    # heavier real queries get 406/429 from an overloaded server.
    _sample = min(MUNICIPALITIES, key=lambda m: m.get("pop_2021", 0))
    test_q = build_overpass_query(_sample)
    working = []
    for ep in CONFIG.overpass_endpoints:
        t0 = time.time()
        try:
            r = requests.post(ep, data={"data": test_q},
                              headers=_DEFAULT_HEADERS, timeout=30)
            dt = time.time() - t0
            if r.status_code == 200:
                working.append(ep)
                if verbose:
                    print(f"  OK  {r.status_code}  {dt:4.1f}s  {urlparse(ep).netloc}")
            elif verbose:
                print(f"  ERR {r.status_code}  {dt:4.1f}s  {urlparse(ep).netloc}")
        except Exception as e:
            if verbose:
                print(f"  ERR ---   {time.time()-t0:4.1f}s  {urlparse(ep).netloc}  "
                      f"{type(e).__name__}")
    if working:
        OVERPASS_ENDPOINTS = working
        OSM_ENABLED = True
        print("\nUsing working Overpass endpoint(s):",
              [urlparse(e).netloc for e in working])
    else:
        OSM_ENABLED = False
        print("\nNo Overpass mirror responded right now -> OSM amenity counts "
              "disabled for this run. The KB will build from Wikipedia + census, and "
              "amenity questions will correctly abstain. Re-run this cell later to "
              "re-enable OSM (cached cities won't re-fetch).")

overpass_health_check()

  OK  200   2.9s  overpass-api.de
  ERR ---   30.5s  overpass.private.coffee  ReadTimeout
  ERR ---   30.7s  overpass.kumi.systems  ReadTimeout

Using working Overpass endpoint(s): ['overpass-api.de']


In [11]:
#@title 4d · Run ingestion -> raw corpus (cached, restart-safe, fail-loud)
CORPUS_PATH = Path(CONFIG.raw_dir) / "corpus.json"

def ingest_all(force: bool = False) -> Dict[str, Any]:
    global OSM_ENABLED
    if CORPUS_PATH.exists() and not force:
        log.info("Loading cached raw corpus from %s", CORPUS_PATH)
        return json.loads(CORPUS_PATH.read_text(encoding="utf-8"))

    # Auto-probe Overpass so a dead/overloaded mirror disables OSM up front
    # instead of stalling the loop with retries on every city.
    try:
        overpass_health_check(verbose=True)
    except Exception as e:
        OSM_ENABLED = False
        log.warning("Overpass health check errored (%s); OSM disabled this run.", e)

    corpus: Dict[str, Any] = {}
    any_wiki = any_osm = any_wv = False
    osm_fail_streak, OSM_FAIL_LIMIT = 0, 3   # circuit breaker for a failing Overpass
    for muni in tqdm(MUNICIPALITIES, desc="Ingesting municipalities"):
        rec: Dict[str, Any] = {"backbone": muni}
        wiki = fetch_wikipedia(muni["wiki"])
        if wiki:
            rec["wikipedia"] = wiki
            any_wiki = True
        wv = fetch_wikivoyage(muni["name"])
        if wv:
            rec["wikivoyage"] = wv
            any_wv = True
        if OSM_ENABLED:
            counts = fetch_overpass(muni)
            rec["osm_counts"] = counts
            if counts:
                any_osm = True
                osm_fail_streak = 0
            else:
                osm_fail_streak += 1
                if osm_fail_streak >= OSM_FAIL_LIMIT:
                    OSM_ENABLED = False
                    log.warning("Overpass failed for %d municipalities in a row; "
                                "disabling OSM for the remaining cities this run "
                                "(re-run §4c-bis + §4d later to backfill).",
                                osm_fail_streak)
        else:
            rec["osm_counts"] = {}
        corpus[muni["name"]] = rec

    if not (any_wiki or any_osm or any_wv):
        raise RuntimeError(
            "INGESTION FAILED: no data retrieved from ANY source for ANY "
            "municipality. Refusing to build a knowledge base from nothing. "
            "Check network access to en.wikipedia.org and overpass-api.de.")
    if not any_wiki:
        log.warning("No Wikipedia text retrieved for any municipality — the KB "
                    "will rely on structured facts only (weaker on 'sentiment'/"
                    "descriptive questions).")

    CORPUS_PATH.write_text(json.dumps(corpus, ensure_ascii=False), encoding="utf-8")
    log.info("Raw corpus persisted to %s (%d municipalities).", CORPUS_PATH, len(corpus))
    return corpus

raw_corpus = ingest_all()
_n_wiki = sum("wikipedia" in r for r in raw_corpus.values())
_n_wv = sum("wikivoyage" in r for r in raw_corpus.values())
_n_osm = sum(1 for r in raw_corpus.values() if r.get("osm_counts"))
log.info("Ingested %d municipalities — %d Wikipedia, %d Wikivoyage, %d with OSM counts.",
         len(raw_corpus), _n_wiki, _n_wv, _n_osm)

  ERR 406   0.5s  overpass-api.de
  ERR ---   30.5s  overpass.private.coffee  ReadTimeout
  ERR ---   30.5s  overpass.kumi.systems  ReadTimeout

No Overpass mirror responded right now -> OSM amenity counts disabled for this run. The KB will build from Wikipedia + census, and amenity questions will correctly abstain. Re-run this cell later to re-enable OSM (cached cities won't re-fetch).


Ingesting municipalities:   0%|          | 0/24 [00:00<?, ?it/s]

## 5 · Document processing

**What:** turn the raw corpus into clean, self-contained **chunks** with metadata.
Three transforms: (1) **clean** Wikipedia text (strip HTML/refs, NFKC-normalize
unicode, collapse whitespace); (2) **normalize structured facts** (backbone +
OSM counts) into readable *fact sentences* that name their municipality explicitly;
(3) **chunk** on sentence boundaries within a token budget, with a small overlap.

**Why this method:** naming the municipality in every fact sentence keeps a chunk
interpretable *in isolation* after retrieval — the retriever may surface a single
chunk with no surrounding context. Sentence-boundary chunking with overlap avoids
cutting a fact in half. Every chunk carries `municipality`, `source`,
`source_title`, and `source_url` so answers can be **cited**.

**Limitations:** we cap each Wikipedia article's kept text to keep chunk counts
free-tier friendly, so very long articles are truncated (intro + early sections,
which are the most general-audience relevant).

In [12]:
#@title Cleaning, fact-sentence, and chunking functions
_TAG_RE = re.compile(r"<[^>]+>")
_WS_RE = re.compile(r"[ \t\u00a0]+")
_MULTINEWLINE_RE = re.compile(r"\n{3,}")
_CITATION_RE = re.compile(r"\[\d+\]")
_WIKI_SECTION_DROP = re.compile(
    r"\n=+\s*(References|External links|See also|Notes|Further reading|"
    r"Bibliography|Sister cities|Twin towns)\s*=+.*", re.IGNORECASE | re.DOTALL)

def clean_text(raw: str) -> str:
    if not raw:
        return ""
    text = _TAG_RE.sub(" ", raw)
    text = html.unescape(text)
    text = unicodedata.normalize("NFKC", text)
    text = _CITATION_RE.sub("", text)
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    lines = [_WS_RE.sub(" ", ln).strip() for ln in text.split("\n")]
    text = "\n".join(lines)
    text = _MULTINEWLINE_RE.sub("\n\n", text)
    return text.strip()

def clean_wikipedia(raw: str, max_chars: int = 9000) -> str:
    text = _WIKI_SECTION_DROP.sub("", raw)          # drop reference-y tail sections
    text = re.sub(r"\n=+\s*[^=\n]+\s*=+\n", "\n", text)  # flatten remaining headers
    text = clean_text(text)
    return text[:max_chars]

def split_sentences(text: str) -> List[str]:
    text = text.strip()
    if not text:
        return []
    parts = re.split(r"(?<=[.!?])\s+(?=[A-Z0-9\"'(])", text)
    return [p.strip() for p in parts if p.strip()]

def approx_token_len(text: str) -> int:
    return int(len(text.split()) / 0.75) + 1

def chunk_text(text: str, *, max_tokens: int, overlap_sentences: int,
               token_len: Callable[[str], int] = approx_token_len) -> List[str]:
    sentences = split_sentences(text)
    if not sentences:
        return []
    chunks, current, current_tokens = [], [], 0
    for sent in sentences:
        st = token_len(sent)
        if st > max_tokens and not current:
            chunks.append(sent); continue
        if current_tokens + st > max_tokens and current:
            chunks.append(" ".join(current))
            current = current[-overlap_sentences:] if overlap_sentences else []
            current_tokens = sum(token_len(s) for s in current)
        current.append(sent); current_tokens += st
    if current:
        chunks.append(" ".join(current))
    deduped = []
    for c in chunks:
        if not deduped or deduped[-1] != c:
            deduped.append(c)
    return deduped

def _fmt_int(n):
    try: return f"{int(round(float(n))):,}"
    except (TypeError, ValueError): return None

def municipality_fact_sentences(m: Dict[str, Any]) -> List[str]:
    name, out = m["name"], []
    pop = _fmt_int(m.get("pop_2021"))
    if pop:
        out.append(f"{name}, Ontario had a population of {pop} in the 2021 Census "
                   f"of Population (Statistics Canada).")
    area = m.get("land_area_km2")
    if pop and area:
        dens = float(m["pop_2021"]) / float(area)
        out.append(f"{name} covers about {float(area):,.1f} square kilometres, a "
                   f"population density of roughly {dens:,.0f} people per square "
                   f"kilometre (2021 Census).")
    if m.get("cd"):
        out.append(f"{name} is located in {m['cd']}, Ontario.")
    if m.get("lat") is not None and m.get("lon") is not None:
        out.append(f"{name} is centred near {float(m['lat']):.3f}° N, "
                   f"{abs(float(m['lon'])):.3f}° W.")
    return out

def amenity_fact_sentences(name, counts, population):
    label = {"park":"parks or green spaces","school":"schools","hospital":"hospitals",
             "pharmacy":"pharmacies","library":"public libraries",
             "transit_stop":"public-transit stops (bus/rail)",
             "supermarket":"supermarkets or grocery stores","restaurant":"restaurants"}
    out = []
    for key, n in counts.items():
        if n is None: continue
        human = label.get(key, key)
        out.append(f"OpenStreetMap has approximately {int(n):,} {human} mapped "
                   f"within {name}'s area (bounding-box approximation).")
        if population and key in ("park","library","school") and n > 0:
            per10k = n / (float(population)/10000.0)
            out.append(f"That is roughly {per10k:,.1f} {human} per 10,000 residents "
                       f"in {name} (OpenStreetMap vs. 2021 Census population).")
    return out

In [13]:
#@title Build chunks with metadata
from dataclasses import dataclass

@dataclass
class Chunk:
    chunk_id: str
    text: str
    municipality: str
    source: str
    source_title: str
    source_url: str
    def to_dict(self): return self.__dict__.copy()
    @staticmethod
    def from_dict(d): return Chunk(**d)

CENSUS_CITATION_URL = ("https://www12.statcan.gc.ca/census-recensement/2021/"
                       "dp-pd/prof/index.cfm?Lang=E")

def build_chunks(corpus: Dict[str, Any]) -> List[Chunk]:
    chunks: List[Chunk] = []
    def add(text, muni, source, title, url):
        for i, piece in enumerate(chunk_text(
                text, max_tokens=CONFIG.chunk_max_tokens,
                overlap_sentences=CONFIG.chunk_overlap_sentences)):
            cid = hashlib.sha1(f"{muni}|{source}|{i}|{piece[:40]}".encode()).hexdigest()[:16]
            chunks.append(Chunk(cid, piece, muni, source, title, url))

    for name, rec in corpus.items():
        m = rec["backbone"]
        # (1) structured facts from the cited 2021-Census backbone
        fact_doc = " ".join(municipality_fact_sentences(m))
        if fact_doc:
            add(fact_doc, name, "statcan-2021-census",
                f"{name} — 2021 Census facts", CENSUS_CITATION_URL)
        # (2) OSM amenity facts
        osm_doc = " ".join(amenity_fact_sentences(name, rec.get("osm_counts", {}),
                                                  m.get("pop_2021")))
        if osm_doc:
            add(osm_doc, name, "openstreetmap",
                f"{name} — OpenStreetMap amenities",
                "https://www.openstreetmap.org/")
        # (3) Wikipedia descriptive text
        wiki = rec.get("wikipedia")
        if wiki:
            add(clean_wikipedia(wiki["text"]), name, "wikipedia",
                wiki["title"], wiki["url"])
        # (4) Wikivoyage editorial "vibe" text (descriptive tone, not reviews)
        wv = rec.get("wikivoyage")
        if wv:
            add(clean_wikipedia(wv["text"]), name, "wikivoyage",
                wv["title"], wv["url"])
    return chunks

CHUNKS = build_chunks(raw_corpus)
if not CHUNKS:
    raise RuntimeError("No chunks produced from corpus — aborting (no fabrication).")
_by_source = pd.Series([c.source for c in CHUNKS]).value_counts()
log.info("Built %d chunks across %d municipalities.", len(CHUNKS),
         len({c.municipality for c in CHUNKS}))
print("Chunks by source:\n", _by_source.to_string())
print("\nExample fact chunk:\n", next(c.text for c in CHUNKS if c.source=="statcan-2021-census"))

Chunks by source:
 wikipedia              278
wikivoyage             266
statcan-2021-census     24

Example fact chunk:
 Toronto, Ontario had a population of 2,794,356 in the 2021 Census of Population (Statistics Canada). Toronto covers about 630.2 square kilometres, a population density of roughly 4,434 people per square kilometre (2021 Census). Toronto is located in Toronto (single-tier), Ontario. Toronto is centred near 43.742° N, 79.373° W.


## 6 · Embedding + FAISS vector store

**What:** embed every chunk with a free `sentence-transformers` model
(`BAAI/bge-small-en-v1.5`, 384-dim) and index them in a **FAISS `IndexFlatIP`**.
Vectors are L2-normalized so inner product equals **cosine similarity**.

**Why this model / index:** BGE-small-v1.5 is a strong, tiny, *ungated* retrieval
model — fast even on CPU, no login required. A flat index gives exact search; over
a few thousand chunks it's instantaneous and avoids approximate-index tuning. BGE
expects a short **instruction prefix on queries only**, which we apply in the
retriever.

**Computational considerations:** embedding is batched and GPU-accelerated when a
GPU is present. The built index + chunk metadata are cached to disk, so re-runs
skip re-embedding entirely (restart-safe).

In [14]:
#@title Embedder (batched, GPU-aware, with fallback) + FAISS store
import faiss
from sentence_transformers import SentenceTransformer

class Embedder:
    """Wraps a sentence-transformers model; BGE query-instruction aware."""
    def __init__(self):
        self.model = None
        self.model_id = None
        for mid in (CONFIG.embed_model, CONFIG.embed_model_fallback):
            try:
                log.info("Loading embedding model: %s", mid)
                self.model = SentenceTransformer(mid, device=DEVICE)
                self.model_id = mid
                break
            except Exception as e:
                log.warning("Failed to load %s (%s); trying fallback.", mid, e)
        if self.model is None:
            raise RuntimeError("Could not load any embedding model.")
        self.dim = self._embedding_dim()
        self._is_bge = "bge" in self.model_id.lower()

    def _embedding_dim(self) -> int:
        """Version-proof embedding-dim lookup (method was renamed across releases)."""
        for meth in ("get_embedding_dimension", "get_sentence_embedding_dimension"):
            fn = getattr(self.model, meth, None)
            if callable(fn):
                d = fn()
                if d:
                    return int(d)
        # Last resort: infer from a probe encode.
        return int(self.model.encode(["probe"], convert_to_numpy=True).shape[1])

    def encode_passages(self, texts: List[str]) -> np.ndarray:
        return self.model.encode(
            texts, batch_size=CONFIG.embed_batch_size, convert_to_numpy=True,
            normalize_embeddings=True, show_progress_bar=len(texts) > 256).astype("float32")

    def encode_query(self, text: str) -> np.ndarray:
        if self._is_bge:
            text = CONFIG.bge_query_instruction + text
        return self.model.encode(
            [text], convert_to_numpy=True, normalize_embeddings=True).astype("float32")

def _l2norm(mat):
    mat = np.asarray(mat, dtype="float32")
    n = np.linalg.norm(mat, axis=1, keepdims=True); n[n == 0] = 1.0
    return mat / n

class VectorStore:
    def __init__(self, dim):
        self.dim = dim; self.index = faiss.IndexFlatIP(dim); self.chunks = []
    def add(self, embeddings, chunks):
        self.index.add(_l2norm(embeddings)); self.chunks.extend(chunks)
    def search(self, query_vec, k, municipality=None, oversample=12):
        if self.index.ntotal == 0: return []
        q = _l2norm(query_vec.reshape(1, -1))
        want = k if municipality is None else min(self.index.ntotal, k*oversample)
        scores, idxs = self.index.search(q, want)
        out = []
        for s, i in zip(scores[0], idxs[0]):
            if i < 0: continue
            ch = self.chunks[i]
            if municipality and ch.municipality.lower() != municipality.lower():
                continue
            out.append({"score": float(s), "chunk": ch})
            if len(out) >= k: break
        return out
    def save(self, ip, mp):
        faiss.write_index(self.index, ip)
        Path(mp).write_text(json.dumps(
            {"dim": self.dim, "model": EMBED_MODEL_ID,
             "chunks": [c.to_dict() for c in self.chunks]}, ensure_ascii=False),
            encoding="utf-8")
    @classmethod
    def load(cls, ip, mp):
        meta = json.loads(Path(mp).read_text(encoding="utf-8"))
        st = cls(meta["dim"]); st.index = faiss.read_index(ip)
        st.chunks = [Chunk.from_dict(d) for d in meta["chunks"]]
        return st

In [15]:
#@title Build (or load cached) FAISS index
INDEX_PATH = str(Path(CONFIG.artifacts_dir) / "faiss.index")
META_PATH = str(Path(CONFIG.artifacts_dir) / "chunks.json")

def build_or_load_store(force: bool = False):
    global EMBED_MODEL_ID
    embedder = Embedder()
    EMBED_MODEL_ID = embedder.model_id
    if Path(INDEX_PATH).exists() and Path(META_PATH).exists() and not force:
        meta = json.loads(Path(META_PATH).read_text(encoding="utf-8"))
        if meta.get("model") == embedder.model_id and len(meta["chunks"]) == len(CHUNKS):
            log.info("Loading cached FAISS index (%d vectors).", len(meta["chunks"]))
            return VectorStore.load(INDEX_PATH, META_PATH), embedder
        log.info("Cache stale (model or chunk-count changed); rebuilding.")
    log.info("Embedding %d chunks with %s ...", len(CHUNKS), embedder.model_id)
    embs = embedder.encode_passages([c.text for c in CHUNKS])
    store = VectorStore(embedder.dim)
    store.add(embs, CHUNKS)
    store.save(INDEX_PATH, META_PATH)
    log.info("Index built + cached: %d vectors, dim=%d.", store.index.ntotal, store.dim)
    return store, embedder

EMBED_MODEL_ID = CONFIG.embed_model
store, embedder = build_or_load_store()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/9 [00:00<?, ?it/s]

## 7 · Retriever

**What:** top-k cosine retrieval with **metadata filtering**. If a query names
exactly one municipality, we restrict retrieval to that municipality's chunks; if
it names two or more (a comparison) or none, we search across all.

**Why:** the single-municipality filter sharply improves precision for questions
like *"What's the population of Guelph?"* — otherwise a near-duplicate fact
sentence from another city can crowd the top-k. For **comparison** queries that name
two or more cities, the retriever instead fetches chunks *per city and merges* them,
so the context is guaranteed to cover every city rather than skewing to one.

**Reranking trade-off (why we don't, by default):** a cross-encoder reranker
(e.g. `BAAI/bge-reranker-base`) would score each (query, chunk) pair jointly and
usually improve ordering — but it means loading a second model and running it over
every candidate, adding latency and VRAM pressure on a free T4 that's already
holding a 3B LLM. For a KB this small the bi-encoder ordering is already strong, so
we expose an optional hook but keep it **off** to guarantee a clean free-tier run.

In [16]:
#@title Retriever
def detect_municipality(query: str, names=MUNI_NAMES) -> Optional[str]:
    q = query.lower()
    hits = [n for n in names if n.lower() in q]
    return hits[0] if len(hits) == 1 else None

def detect_municipalities(query: str, names=MUNI_NAMES) -> List[str]:
    """All municipalities named in the query (for comparison / multi-city queries)."""
    q = query.lower()
    return [n for n in names if n.lower() in q]

class Retriever:
    def __init__(self, store, embedder):
        self.store, self.embedder = store, embedder
    def retrieve(self, query: str, k: int = None, municipality: str = "auto"):
        k = k or CONFIG.top_k
        qv = self.embedder.encode_query(query)

        # Comparison queries name 2+ municipalities: retrieve per city and merge so
        # the context covers EVERY named city instead of skewing to one of them.
        if municipality == "auto":
            named = detect_municipalities(query)
            if len(named) >= 2:
                per = max(2, math.ceil(k / len(named)))
                merged, seen = [], set()
                for m in named:
                    hits = self.store.search(qv, k=per, municipality=m)
                    # Guarantee the authoritative 2021-Census fact card for each
                    # named city is present, so a fuzzy comparison query never
                    # grounds population on a stale descriptive figure that merely
                    # ranked higher. (Structured-fact injection for named entities.)
                    census = next((c for c in self.store.chunks
                                   if c.municipality == m
                                   and c.source == "statcan-2021-census"), None)
                    if census and all(h["chunk"].chunk_id != census.chunk_id for h in hits):
                        hits.append({"score": 0.0, "chunk": census})
                    for r in hits:
                        if r["chunk"].chunk_id not in seen:
                            seen.add(r["chunk"].chunk_id); merged.append(r)
                merged.sort(key=lambda r: r["score"], reverse=True)
                return merged
            municipality = named[0] if named else None

        results = self.store.search(qv, k=k, municipality=municipality)
        # If a municipality filter yields too few hits, retry unfiltered so we
        # never starve the generator of context.
        if municipality and len(results) < k:
            extra = self.store.search(qv, k=k, municipality=None)
            seen = {r["chunk"].chunk_id for r in results}
            for r in extra:
                if r["chunk"].chunk_id not in seen:
                    results.append(r)
                if len(results) >= k: break
        return results

retriever = Retriever(store, embedder)

# Smoke test retrieval (no LLM needed).
_demo = retriever.retrieve("What is the population of Guelph?", k=3)
print("Query: What is the population of Guelph?")
for r in _demo:
    print(f"  [{r['score']:.3f}] ({r['chunk'].municipality}/{r['chunk'].source}) "
          f"{r['chunk'].text[:90]}...")

Query: What is the population of Guelph?
  [0.869] (Guelph/statcan-2021-census) Guelph, Ontario had a population of 143,740 in the 2021 Census of Population (Statistics C...
  [0.785] (Guelph/wikipedia) Guelph ( GWELF; 2021 Canadian Census population 143,740) is a city in Southwestern Ontario...
  [0.749] (Guelph/wikivoyage) Guelph is a city of 132,000 people (2016) in Southwestern Ontario on the banks of the Spee...


## 8 · Generator (small open instruct LLM)

**What:** load a small **ungated** instruct LLM and generate a **grounded, cited**
answer from the retrieved chunks. Fallback chain:
`Qwen2.5-3B-Instruct` → `Phi-4-mini-instruct` → `Qwen2.5-1.5B-Instruct`.

**Why these:** all three are open and **ungated** (no HF token / licence click),
so the notebook needs no auth. Qwen2.5-3B in fp16 is ~6 GB and fits a free T4;
the 1.5B fallback keeps things runnable on tighter memory or CPU. (Llama-3.2-3B is
deliberately excluded — it's gated and would break the no-auth constraint.)

**Version-proofing:** `transformers` renames/removes kwargs across releases (e.g.
`torch_dtype` → `dtype`). Instead of pinning a version, we **introspect the
installed signatures** and pass only supported kwargs, remapping renamed keys.

**Grounding:** a strict system prompt instructs the model to answer *only* from the
provided context, cite sources inline as `[S1]`, `[S2]`, and explicitly say
*"I don't have data on that"* when the context is insufficient — never inventing
statistics.

In [17]:
#@title Load generator with signature-safe kwargs + fallback chain
def filter_kwargs_for(func, candidate, rename=None):
    """Pass only kwargs `func` accepts; apply {old:new} renames first."""
    rename = rename or {}
    remapped = {rename.get(k, k): v for k, v in candidate.items()}
    try:
        sig = inspect.signature(func)
    except (TypeError, ValueError):
        return dict(remapped)
    params = sig.parameters.values()
    if any(p.kind == inspect.Parameter.VAR_KEYWORD for p in params):
        return dict(remapped)
    allowed = {p.name for p in params if p.kind in (
        inspect.Parameter.POSITIONAL_OR_KEYWORD, inspect.Parameter.KEYWORD_ONLY)}
    return {k: v for k, v in remapped.items() if k in allowed}

GEN = {"model": None, "tokenizer": None, "model_id": None}

def load_generator():
    if not _HAS_TORCH:
        log.warning("Torch unavailable — generation disabled (retrieval-only mode).")
        return
    from transformers import AutoModelForCausalLM, AutoTokenizer
    use_gpu = (DEVICE == "cuda")
    model_chain = CONFIG.gen_models if use_gpu else CONFIG.gen_models_cpu
    if not use_gpu:
        log.warning("No GPU detected: using CPU-friendly generator chain %s with "
                    "shorter answers. Generation still works but is slower than a T4.",
                    list(model_chain))
    for mid in model_chain:
        try:
            log.info("Loading generator: %s", mid)
            tok = AutoTokenizer.from_pretrained(mid)
            # Handle the torch_dtype -> dtype rename robustly.
            load_kwargs = {"low_cpu_mem_usage": True}
            if use_gpu:
                load_kwargs["device_map"] = "auto"
            model = None
            for dtype_key in ("dtype", "torch_dtype"):
                try:
                    kw = dict(load_kwargs)
                    if use_gpu:
                        kw[dtype_key] = torch.float16
                    model = AutoModelForCausalLM.from_pretrained(mid, **kw)
                    break
                except TypeError:
                    continue
            if model is None:  # last resort: minimal kwargs
                model = AutoModelForCausalLM.from_pretrained(mid)
            if not use_gpu:
                model = model.to("cpu")
            model.eval()
            if tok.pad_token_id is None:
                tok.pad_token = tok.eos_token
            GEN.update(model=model, tokenizer=tok, model_id=mid)
            log.info("Generator ready: %s", mid)
            return
        except Exception as e:
            log.warning("Could not load %s (%s); trying next fallback.", mid, e)
    log.error("No generator model could be loaded; answers will be extractive only.")

load_generator()

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [18]:
#@title Grounding prompt + signature-safe generation
SYSTEM_PROMPT = (
    "You are a careful research assistant answering questions about living in "
    "Ontario municipalities. Follow these rules strictly:\n"
    "1. Answer ONLY using facts in the provided CONTEXT. Do not use outside "
    "knowledge and never invent numbers or statistics.\n"
    "2. Cite the source of each fact inline using its tag, e.g. [S1], [S2].\n"
    "3. If the context does not contain enough information to answer, say exactly: "
    "\"I don't have data on that in my knowledge base.\" Do not guess.\n"
    "4. Be concise, neutral, and descriptive. Report what the data says; do not "
    "advise the user where to live."
)

def build_context(results) -> Tuple[str, List[Dict[str, Any]]]:
    """Number retrieved chunks as [S1..] and return (context_text, source_table)."""
    lines, sources = [], []
    for i, r in enumerate(results, 1):
        ch = r["chunk"]
        tag = f"S{i}"
        lines.append(f"[{tag}] ({ch.municipality} — {ch.source}) {ch.text}")
        sources.append({"tag": tag, "municipality": ch.municipality,
                        "source": ch.source, "title": ch.source_title,
                        "url": ch.source_url, "score": round(r["score"], 3),
                        "chunk_id": ch.chunk_id})
    return "\n\n".join(lines), sources

# Valid GenerationConfig fields on the installed transformers — used to filter.
def _valid_gen_fields():
    try:
        from transformers import GenerationConfig
        return set(vars(GenerationConfig()))
    except Exception:
        return set()

_GEN_FIELDS = _valid_gen_fields()

def generate_answer(question: str, context: str) -> str:
    if GEN["model"] is None:
        return ("[generation unavailable] Based on retrieved context only. "
                "See the cited sources below.")
    tok, model = GEN["tokenizer"], GEN["model"]
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"CONTEXT:\n{context}\n\nQUESTION: {question}"},
    ]
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt, return_tensors="pt").to(model.device)

    _max_new = (CONFIG.gen_max_new_tokens if DEVICE == "cuda"
                else CONFIG.gen_max_new_tokens_cpu)
    candidate = {"max_new_tokens": _max_new, "do_sample": False,
                 "repetition_penalty": 1.1, "num_beams": 1,
                 "pad_token_id": tok.pad_token_id}
    gen_kwargs = ({k: v for k, v in candidate.items() if k in _GEN_FIELDS}
                  if _GEN_FIELDS else candidate)
    gen_kwargs["pad_token_id"] = tok.pad_token_id  # always keep this one
    with torch.no_grad():
        out = model.generate(**inputs, **gen_kwargs)
    new_tokens = out[0][inputs["input_ids"].shape[1]:]
    return tok.decode(new_tokens, skip_special_tokens=True).strip()

## 9 · Query interface

**What:** a single `answer(question)` function that retrieves, builds the grounded
context, generates a cited answer, and returns both the answer text and the table
of cited sources. A small interactive loop (`ipywidgets` when available, plain
`input()` otherwise) lets you ask ad-hoc questions.

In [19]:
#@title answer() + pretty printer
@dataclass
class RagResponse:
    question: str
    answer: str
    sources: List[Dict[str, Any]]
    municipality_filter: Optional[str]

def answer(question: str, k: int = None, municipality: str = "auto") -> RagResponse:
    """Retrieve -> ground -> generate. Returns answer + cited sources."""
    results = retriever.retrieve(question, k=k, municipality=municipality)
    used_filter = detect_municipality(question) if municipality == "auto" else municipality
    if not results:
        return RagResponse(question,
                           "I don't have data on that in my knowledge base.",
                           [], used_filter)
    context, sources = build_context(results)
    ans = generate_answer(question, context)
    return RagResponse(question, ans, sources, used_filter)

def show(resp: RagResponse) -> None:
    print("Q:", resp.question)
    if resp.municipality_filter:
        print(f"   (retrieval filtered to: {resp.municipality_filter})")
    print("\nA:", resp.answer)
    print("\nCited sources:")
    for s in resp.sources:
        print(f"  [{s['tag']}] {s['title']} — {s['source']} "
              f"(sim={s['score']}) {s['url']}")

# One end-to-end demo (will actually call the LLM if it loaded).
show(answer("What is the population of Guelph and what county is it in?"))

Q: What is the population of Guelph and what county is it in?
   (retrieval filtered to: Guelph)

A: The population of Guelph is 143,740 according to the 2021 Census of Population. Guelph is located in Wellington County, Ontario.

Cited sources:
  [S1] Guelph — 2021 Census facts — statcan-2021-census (sim=0.834) https://www12.statcan.gc.ca/census-recensement/2021/dp-pd/prof/index.cfm?Lang=E
  [S2] Guelph — wikipedia (sim=0.79) https://en.wikipedia.org/wiki/Guelph
  [S3] Guelph — wikivoyage (sim=0.738) https://en.wikivoyage.org/wiki/Guelph
  [S4] Guelph — wikipedia (sim=0.703) https://en.wikipedia.org/wiki/Guelph
  [S5] Guelph — wikivoyage (sim=0.692) https://en.wikivoyage.org/wiki/Guelph


In [20]:
#@title Optional interactive loop
def interactive():
    try:
        import ipywidgets as w
        from IPython.display import display, clear_output
        box = w.Text(placeholder="Ask about an Ontario municipality...",
                     layout=w.Layout(width="80%"))
        btn = w.Button(description="Ask", button_style="primary")
        out = w.Output()
        def on_click(_):
            with out:
                clear_output()
                if box.value.strip():
                    show(answer(box.value.strip()))
        btn.on_click(on_click); box.on_submit(lambda _: on_click(None))
        display(w.HBox([box, btn]), out)
    except Exception:
        print("ipywidgets unavailable; using text loop. Type 'quit' to stop.")
        while True:
            try:
                q = input("\nQuestion> ").strip()
            except (EOFError, KeyboardInterrupt):
                break
            if q.lower() in {"quit", "exit", ""}:
                break
            show(answer(q))

# Uncomment to use interactively:
# interactive()

## 10 · Evaluation harness

This is what makes the project credible. We evaluate three things against the
**actual ingested data**:

1. **Retrieval hit-rate / recall@k** — for each eval question we specify the
   *municipality* and *source type* whose chunk should be retrieved, and check
   whether such a chunk appears in the top-k.
2. **Answer faithfulness / groundedness** — two reproducible checks: a
   **lexical-overlap** score (fraction of the answer's content words present in the
   retrieved context) and a **numeric-support** check (every number in the answer
   must also appear in the context — the highest-risk failure mode). An optional
   **LLM-as-judge** using the *same local model* is included and clearly labelled.
3. **Abstention behaviour** — on deliberately out-of-scope questions, does the
   system correctly say *"I don't have data on that"* instead of hallucinating?

**Why both lexical and LLM judges:** the lexical/numeric checks are deterministic,
cheap, and can't themselves hallucinate; the LLM judge is more semantically aware
but is the same small model being evaluated, so we report it as a *soft* signal
only.

In [21]:
#@title Evaluation metrics
_WORD_RE = re.compile(r"[a-z0-9]+")
_STOP = {"the","a","an","and","or","of","to","in","on","for","is","are","was",
         "were","with","at","by","as","that","this","it","its","be","has","have",
         "had","about","from","than","roughly","approximately","per","there",
         "their","which","what","does","do","near"}
def _content_tokens(t):
    return [x for x in _WORD_RE.findall(t.lower()) if x not in _STOP and len(x) > 2]

def recall_at_k(retrieved_ids, relevant_ids, k):
    rel = set(relevant_ids)
    if not rel: return float("nan")
    return len(set(retrieved_ids[:k]) & rel) / len(rel)

def lexical_faithfulness(answer_text, context):
    a = _content_tokens(answer_text)
    if not a: return float("nan")
    ctx = set(_content_tokens(context))
    return sum(1 for t in a if t in ctx) / len(a)

_NUM_RE = re.compile(r"\d[\d,]*(?:\.\d+)?")
def numeric_claims_supported(answer_text, context):
    norm = lambda s: (s.replace(",", "").rstrip(".0") or "0")
    ans = [norm(m.group()) for m in _NUM_RE.finditer(answer_text)]
    ctx = {norm(m.group()) for m in _NUM_RE.finditer(context)}
    if not ans: return (0, 0)
    return (sum(1 for n in ans if n in ctx), len(ans))

ABSTAIN_MARKERS = ("i don't have data","i do not have data","don't have enough",
                   "insufficient","no data","not enough information","cannot answer")
def looks_like_abstention(a):
    a = a.lower(); return any(m in a for m in ABSTAIN_MARKERS)

def matches_expectation(chunk, expect):
    ok = True
    if expect.get("municipality"):
        ok &= chunk.municipality.lower() == expect["municipality"].lower()
    if expect.get("source"):
        ok &= chunk.source == expect["source"]
    return ok

In [22]:
#@title Evaluation question set (grounded in the ingested data)
# Factual questions carry an `expect` of {municipality, source} identifying which
# KB chunk *should* be retrieved. Abstain questions should trigger "I don't have data".
def _mk(question, muni=None, source=None, qtype="factual"):
    return {"question": question, "type": qtype,
            "expect": ({"municipality": muni, "source": source}
                       if qtype == "factual" else None)}

_FACT_CITIES = [m for m in ["Guelph","Kingston","Hamilton","Ottawa","Waterloo",
                            "London","Barrie","Kitchener"] if m in MUNI_NAMES]

EVAL_QUESTIONS: List[Dict[str, Any]] = []
for c in _FACT_CITIES:
    EVAL_QUESTIONS.append(_mk(f"What was the population of {c} in the 2021 Census?", c, "statcan-2021-census"))
    EVAL_QUESTIONS.append(_mk(f"Which census division or region is {c} in?", c, "statcan-2021-census"))
for c in [x for x in ["Guelph","Kingston","Hamilton","Ottawa","London","Barrie"] if x in MUNI_NAMES]:
    EVAL_QUESTIONS.append(_mk(f"Roughly how many parks are mapped in {c}?", c, "openstreetmap"))
for c in [x for x in ["Waterloo","Kitchener","Windsor","Guelph"] if x in MUNI_NAMES]:
    EVAL_QUESTIONS.append(_mk(f"How many public libraries does OpenStreetMap record in {c}?", c, "openstreetmap"))
for c in [x for x in ["Hamilton","Kingston","Ottawa","London","Guelph","Windsor"] if x in MUNI_NAMES]:
    EVAL_QUESTIONS.append(_mk(f"What is {c} known for historically or economically?", c, "wikipedia"))

# Out-of-scope / abstention questions (data intentionally NOT in the KB).
for q in [
    "What is the median rent for a two-bedroom apartment in Guelph?",
    "What is the crime rate in Kingston compared to the national average?",
    "How good are the public schools in Hamilton by test scores?",
    "What is the population of Vancouver, British Columbia?",
    "Which Ontario city has the best nightlife?",
    "What will housing prices in Barrie be in 2030?",
]:
    EVAL_QUESTIONS.append(_mk(q, qtype="abstain"))

# If OpenStreetMap was unavailable this run, amenity questions are genuinely
# unanswerable -> reclassify them as expected-abstain so the false-abstention
# metric reflects only truly answerable questions (honest evaluation, not gaming).
_has_osm = any(c.source == "openstreetmap" for c in CHUNKS)
if not _has_osm:
    _conv = 0
    for q in EVAL_QUESTIONS:
        if q["type"] == "factual" and q["expect"] and q["expect"].get("source") == "openstreetmap":
            q["type"] = "abstain"; q["expect"] = None; _conv += 1
    if _conv:
        log.info("No OSM data this run: reclassified %d amenity questions as "
                 "expected-abstain (unanswerable without OpenStreetMap).", _conv)

# On CPU, trim to a balanced subset so evaluation finishes in minutes, not hours.
if DEVICE != "cuda" and len(EVAL_QUESTIONS) > CONFIG.eval_max_questions_cpu:
    import random as _r; _r.seed(CONFIG.seed)
    _fact = [q for q in EVAL_QUESTIONS if q["type"] == "factual"]
    _abst = [q for q in EVAL_QUESTIONS if q["type"] == "abstain"]
    _na = min(4, len(_abst)); _nf = CONFIG.eval_max_questions_cpu - _na
    EVAL_QUESTIONS = _r.sample(_fact, min(_nf, len(_fact))) + _r.sample(_abst, _na)
    log.info("CPU mode: trimmed evaluation to %d questions for speed "
             "(raise CONFIG.eval_max_questions_cpu or use a GPU for the full set).",
             len(EVAL_QUESTIONS))

log.info("Evaluation set: %d questions (%d factual, %d abstention).",
         len(EVAL_QUESTIONS),
         sum(q["type"]=="factual" for q in EVAL_QUESTIONS),
         sum(q["type"]=="abstain" for q in EVAL_QUESTIONS))

In [23]:
#@title Optional LLM-as-judge (same local model; soft signal only)
RUN_LLM_JUDGE = False  #@param {type:"boolean"}

def llm_judge_supported(answer_text: str, context: str) -> Optional[bool]:
    """Ask the same local model whether the answer is fully supported by context.

    Clearly a SOFT signal: it is the same small model under evaluation, so treat
    it as indicative, not authoritative.
    """
    if GEN["model"] is None:
        return None
    tok, model = GEN["tokenizer"], GEN["model"]
    msg = [
        {"role": "system", "content": "You are a strict grading assistant. Reply with only YES or NO."},
        {"role": "user", "content": (f"CONTEXT:\n{context}\n\nANSWER:\n{answer_text}\n\n"
                                      "Is every factual claim in ANSWER directly supported by CONTEXT? "
                                      "Reply YES or NO only.")},
    ]
    prompt = tok.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=4, do_sample=False,
                             pad_token_id=tok.pad_token_id)
    txt = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return txt.strip().upper().startswith("YES")

In [24]:
#@title Run evaluation
def run_eval(questions=EVAL_QUESTIONS, k=None):
    k = k or CONFIG.top_k
    rows = []
    for item in tqdm(questions, desc="Evaluating"):
        q, qtype, expect = item["question"], item["type"], item["expect"]
        results = retriever.retrieve(q, k=k)
        retrieved_ids = [r["chunk"].chunk_id for r in results]
        context, _ = build_context(results) if results else ("", [])
        resp = answer(q, k=k)

        row = {"question": q, "type": qtype, "n_retrieved": len(results)}
        if qtype == "factual":
            hit = int(any(matches_expectation(r["chunk"], expect) for r in results))
            row["hit@k"] = hit
            row["expect_muni"] = expect.get("municipality")
            row["expect_source"] = expect.get("source")
            row["lex_faith"] = round(lexical_faithfulness(resp.answer, context), 3)
            sup, tot = numeric_claims_supported(resp.answer, context)
            row["num_supported"] = sup; row["num_total"] = tot
            row["num_support_rate"] = round(sup/tot, 3) if tot else None
            row["abstained"] = int(looks_like_abstention(resp.answer))
            if RUN_LLM_JUDGE:
                j = llm_judge_supported(resp.answer, context)
                row["llm_judge_supported"] = None if j is None else int(j)
        else:  # abstain
            row["abstained_correctly"] = int(looks_like_abstention(resp.answer))
        row["answer"] = resp.answer[:300]
        rows.append(row)
    return pd.DataFrame(rows)

eval_df = run_eval()

# ---- aggregate ------------------------------------------------------------ #
fact = eval_df[eval_df["type"] == "factual"]
abst = eval_df[eval_df["type"] == "abstain"]
summary = {
    "n_questions": len(eval_df),
    "n_factual": len(fact),
    "n_abstain": len(abst),
    f"retrieval_hit@{CONFIG.top_k}": round(fact["hit@k"].mean(), 3) if len(fact) else None,
    "mean_lexical_faithfulness": round(fact["lex_faith"].mean(), 3) if len(fact) else None,
    "numeric_support_rate": (round(fact["num_supported"].sum()/fact["num_total"].sum(), 3)
                             if len(fact) and fact["num_total"].sum() else None),
    "factual_false_abstention_rate": round(fact["abstained"].mean(), 3) if len(fact) else None,
    "abstention_accuracy_on_oos": round(abst["abstained_correctly"].mean(), 3) if len(abst) else None,
}
print("\n===== EVALUATION SUMMARY =====")
for k_, v in summary.items():
    print(f"  {k_}: {v}")

Evaluating:   0%|          | 0/38 [00:00<?, ?it/s]


===== EVALUATION SUMMARY =====
  n_questions: 38
  n_factual: 22
  n_abstain: 16
  retrieval_hit@5: 1.0
  mean_lexical_faithfulness: 0.785
  numeric_support_rate: 1.0
  factual_false_abstention_rate: 0.045
  abstention_accuracy_on_oos: 1.0


In [25]:
#@title Persist evaluation report (CSV + markdown)
EVAL_CSV = Path(CONFIG.artifacts_dir) / "eval_results.csv"
EVAL_MD = Path(CONFIG.artifacts_dir) / "eval_report.md"
eval_df.to_csv(EVAL_CSV, index=False)

def _md_table(d: Dict[str, Any]) -> str:
    lines = ["| metric | value |", "|---|---|"]
    for k_, v in d.items():
        lines.append(f"| {k_} | {v} |")
    return "\n".join(lines)

EVAL_MD.write_text(
    "# Evaluation Report\n\n"
    f"Embedding model: `{EMBED_MODEL_ID}`  \n"
    f"Generator: `{GEN['model_id']}`  \n"
    f"Chunks indexed: {store.index.ntotal}  \n"
    f"top_k: {CONFIG.top_k}\n\n"
    "## Summary metrics\n\n" + _md_table(summary) + "\n\n"
    "## Metric definitions\n\n"
    f"- **retrieval_hit@{CONFIG.top_k}**: fraction of factual questions where a chunk "
    "of the expected municipality+source appeared in the top-k.\n"
    "- **mean_lexical_faithfulness**: mean fraction of answer content-words found in "
    "retrieved context (deterministic groundedness proxy).\n"
    "- **numeric_support_rate**: fraction of numbers in answers that also appear in "
    "context (guards against fabricated statistics).\n"
    "- **factual_false_abstention_rate**: how often the system wrongly abstained on "
    "answerable questions (lower is better).\n"
    "- **abstention_accuracy_on_oos**: fraction of out-of-scope questions correctly "
    "answered with 'I don't have data' (higher is better).\n",
    encoding="utf-8")
log.info("Wrote %s and %s", EVAL_CSV, EVAL_MD)
eval_df.head(12)

,question,type,n_retrieved,hit@k,expect_muni,expect_source,lex_faith,num_supported,num_total,num_support_rate,abstained,answer,abstained_correctly
0,What was the population of Guelph in the 2021 ...,factual,5,1.0,Guelph,statcan-2021-census,1.000,3.0,3.0,1.0,0.0,"In the 2021 Census, Guelph, Ontario had a popu...",NaN
1,Which census division or region is Guelph in?,factual,5,1.0,Guelph,statcan-2021-census,0.750,1.0,1.0,1.0,0.0,Guelph is in the census division of Wellington...,NaN
2,What was the population of Kingston in the 202...,factual,5,1.0,Kingston,statcan-2021-census,0.875,3.0,3.0,1.0,0.0,"According to [S1], the population of Kingston,...",NaN
3,Which census division or region is Kingston in?,factual,5,1.0,Kingston,statcan-2021-census,1.000,1.0,1.0,1.0,0.0,Kingston is in the Census Division of Eastern ...,NaN
4,What was the population of Hamilton in the 202...,factual,5,1.0,Hamilton,statcan-2021-census,1.000,3.0,3.0,1.0,0.0,The population of Hamilton in the 2021 Census ...,NaN
5,Which census division or region is Hamilton in?,factual,5,1.0,Hamilton,statcan-2021-census,0.857,1.0,1.0,1.0,0.0,Hamilton is in the geographic division of Hami...,NaN
6,What was the population of Ottawa in the 2021 ...,factual,5,1.0,Ottawa,statcan-2021-census,1.000,3.0,3.0,1.0,0.0,The population of Ottawa in the 2021 Census wa...,NaN
7,Which census division or region is Ottawa in?,factual,5,1.0,Ottawa,statcan-2021-census,0.944,2.0,2.0,1.0,0.0,Ottawa is in the National Capital Region (NCR)...,NaN
8,What was the population of Waterloo in the 202...,factual,5,1.0,Waterloo,statcan-2021-census,0.875,3.0,3.0,1.0,0.0,"According to [S1], the population of Waterloo,...",NaN
9,Which census division or region is Waterloo in?,factual,5,1.0,Waterloo,statcan-2021-census,1.000,1.0,1.0,1.0,0.0,"Waterloo is in the Waterloo Region, Ontario. [S2]",NaN


## 10b · Qualitative examples: a good answer, a caught abstention, a failure

Numbers alone don't tell the story. Below we show one **good grounded answer**, one
**correct abstention** on out-of-scope data, and one **failure case** with analysis
— because being honest about where the system breaks is part of the deliverable.

In [26]:
#@title Three illustrative cases
print("="*70, "\nCASE 1 — GOOD GROUNDED ANSWER\n", "="*70)
show(answer("What was Kingston's 2021 population and what is it historically known for?"))

print("\n" + "="*70, "\nCASE 2 — CORRECT ABSTENTION (out-of-scope)\n", "="*70)
show(answer("What is the average monthly rent in Hamilton?"))

print("\n" + "="*70, "\nCASE 3 — LIKELY FAILURE CASE (analysis below)\n", "="*70)
# Comparisons stress the small model: it must synthesize facts about TWO cities
# retrieved without a single-municipality filter. Watch for dropped citations or
# a fact attributed to the wrong city.
show(answer("Compare Guelph and Kingston for a young family."))
print("\nANALYSIS: comparison questions name 2+ cities, so the retriever now fetches "
      "chunks PER city and merges them (see §7), guaranteeing both are represented "
      "in context, and each city's 2021-Census fact card is force-included so "
      "population reflects the census, not a stale descriptive figure. The remaining "
      "risk is the small model blurring which fact belongs to which city. Further "
      "mitigations: a cross-encoder reranker, or summarize each city then compare.")

CASE 1 — GOOD GROUNDED ANSWER
Q: What was Kingston's 2021 population and what is it historically known for?
   (retrieval filtered to: Kingston)

A: According to [S1], Kingston's population in 2021 was 132,485.

[From S2] Kingston is historically known for its rich history, featuring numerous churches, old buildings, picturesque neighborhoods, and 19th-century fortifications. It is often referred to as the "Limestone City" due to the many heritage buildings constructed using local limestone.

Cited sources:
  [S1] Kingston — 2021 Census facts — statcan-2021-census (sim=0.756) https://www12.statcan.gc.ca/census-recensement/2021/dp-pd/prof/index.cfm?Lang=E
  [S2] Kingston (Ontario) — wikivoyage (sim=0.735) https://en.wikivoyage.org/wiki/Kingston_(Ontario)
  [S3] Kingston, Ontario — wikipedia (sim=0.687) https://en.wikipedia.org/wiki/Kingston,_Ontario
  [S4] Kingston, Ontario — wikipedia (sim=0.677) https://en.wikipedia.org/wiki/Kingston,_Ontario
  [S5] Kingston, Ontario — wikipedia (sim=

## 11 · Generate `README.md` and `requirements.txt`

We write both deliverables from *this run's* real artifacts — the README embeds the
actual evaluation numbers and a real example Q&A, so it never drifts from what the
notebook produced.

In [27]:
#@title Write requirements.txt
REQUIREMENTS = """\
# Ontario Places-to-Live RAG Assistant — pinned-compatible, not over-constrained.
sentence-transformers>=2.7.0
faiss-cpu>=1.7.4
transformers>=4.44.0
accelerate>=0.33.0
torch            # provided by Colab; listed for completeness
requests>=2.31.0
tqdm>=4.66.0
pandas>=2.0.0
numpy>=1.24.0
ipywidgets>=8.0.0
"""
Path("requirements.txt").write_text(REQUIREMENTS, encoding="utf-8")
(Path(CONFIG.artifacts_dir) / "requirements.txt").write_text(REQUIREMENTS, encoding="utf-8")
print(REQUIREMENTS)

# Ontario Places-to-Live RAG Assistant — pinned-compatible, not over-constrained.
sentence-transformers>=2.7.0
faiss-cpu>=1.7.4
transformers>=4.44.0
accelerate>=0.33.0
torch            # provided by Colab; listed for completeness
requests>=2.31.0
tqdm>=4.66.0
pandas>=2.0.0
numpy>=1.24.0
ipywidgets>=8.0.0



In [28]:
#@title Write README.md (embeds this run's real results)
_example = answer("What is the population of Guelph and what county is it in?")
_example_sources = "\n".join(
    f"  - [{s['tag']}] {s['title']} ({s['source']}) — {s['url']}"
    for s in _example.sources) or "  (none)"

_arch = r'''
REAL SOURCES → INGEST (cache/rate-limit/backoff) → PROCESS (clean/fact-sentence/chunk)
      → EMBED (BGE-small-v1.5) → FAISS (cosine) → RETRIEVE (+muni filter)
      → GROUNDING PROMPT → Qwen2.5-3B-Instruct → CITED ANSWER
      → EVAL (recall@k · faithfulness · numeric-support · abstention)
'''

README = f"""# 🏙️ Ontario Places-to-Live RAG Assistant

A retrieval-augmented generation (RAG) system that answers questions about living
in Ontario municipalities using **only real, public data**, and generates
**cited, source-grounded** answers with a small open LLM that runs **free in Google
Colab**. It is deliberately *descriptive and sourced, not prescriptive*: it reports
what the data says and abstains when the data is silent.

## Architecture
```{_arch}```

## Real data sources (all free, no auth)
- **Wikipedia** (Action API `prop=extracts`) — descriptive text per municipality.
- **Wikivoyage** (same MediaWiki API) — editorial, qualitative "what's it like" prose;
  labelled as travel-writer tone, not resident reviews or measured sentiment.
- **OpenStreetMap Overpass API** — counts of mapped amenities (parks, schools,
  hospitals, libraries, pharmacies, transit stops, supermarkets, restaurants) within
  a bounding box per city.
- **Statistics Canada, 2021 Census of Population** — municipality populations, land
  area, and density (curated backbone; a live StatCan WDS `getFullTableDownloadCSV`
  client is also wired in and off by default).
- **Sentiment**: not included unless you supply a cited per-municipality CSV.

## Models
- Embeddings: `{EMBED_MODEL_ID}` (ungated).
- Generator: `{GEN['model_id']}` with fallback chain
  Qwen2.5-3B-Instruct → Phi-4-mini-instruct → Qwen2.5-1.5B-Instruct (all ungated).

## How to run (fresh Colab)
1. Upload `Ontario_Places_to_Live_RAG.ipynb` to Google Colab.
2. `Runtime → Change runtime type → GPU` (T4 is enough; CPU also works, slower).
3. `Runtime → Run all`. First run ingests + embeds (cached afterwards).
4. Ask questions with `answer("...")` or call `interactive()`.

## Example (from this run)
**Q:** {_example.question}

**A:** {_example.answer}

**Cited sources:**
{_example_sources}

## Evaluation (this run)
| metric | value |
|---|---|
""" + "\n".join(f"| {k_} | {v} |" for k_, v in summary.items()) + f"""

Full per-question results: `artifacts/eval_results.csv`; report:
`artifacts/eval_report.md`.

## Saved artifacts
- `artifacts/faiss.index` — the FAISS vector index.
- `artifacts/chunks.json` — chunk text + metadata (the processed chunk store).
- `artifacts/eval_results.csv`, `artifacts/eval_report.md` — evaluation outputs.
- `raw/corpus.json` — the raw ingested corpus; `http_cache/` — cached API responses.

## Honest limitations
- **Retrieval can miss**: if the relevant chunk isn't in the top-k, the model can't use it.
- **Small local model**: a 3B model is a weaker synthesizer than frontier models and
  can still misread context, especially on multi-city comparisons.
- **OSM counts are bounding-box approximations** and depend on mapping completeness —
  treat them as rough *relative* signals, not exact inventories.
- **Census scope**: "population" is the city proper (census subdivision), not the
  metro area; cross-source boundaries don't always align.
- **No cost/rent/crime/school-quality data by default** — the system correctly
  abstains on those rather than guessing.

## Future work
Per-city retrieval-then-merge for comparisons; a cross-encoder reranker;
wiring StatCan WDS income/housing tables end-to-end; adding a cited sentiment layer;
hybrid (BM25 + dense) retrieval; answer-level citation verification.

## References
- Wikimedia REST/Action API: https://www.mediawiki.org/wiki/API:REST_API
- OpenStreetMap Overpass API: https://wiki.openstreetmap.org/wiki/Overpass_API
- Statistics Canada Web Data Service: https://www.statcan.gc.ca/en/developers/wds
- BGE embeddings: https://huggingface.co/BAAI/bge-small-en-v1.5
- Qwen2.5: https://huggingface.co/Qwen/Qwen2.5-3B-Instruct
- FAISS: https://github.com/facebookresearch/faiss

## License / data attribution
Code: MIT. Data belongs to its providers and is used under their terms —
Wikipedia text (CC BY-SA), OpenStreetMap (ODbL), Statistics Canada (Statistics
Canada Open Licence). This tool is informational and not advice.
"""
Path("README.md").write_text(README, encoding="utf-8")
(Path(CONFIG.artifacts_dir) / "README.md").write_text(README, encoding="utf-8")
log.info("Wrote README.md (%d chars) and requirements.txt.", len(README))
print(README[:1200], "\n...[truncated]")

# 🏙️ Ontario Places-to-Live RAG Assistant

A retrieval-augmented generation (RAG) system that answers questions about living
in Ontario municipalities using **only real, public data**, and generates
**cited, source-grounded** answers with a small open LLM that runs **free in Google
Colab**. It is deliberately *descriptive and sourced, not prescriptive*: it reports
what the data says and abstains when the data is silent.

## Architecture
```
REAL SOURCES → INGEST (cache/rate-limit/backoff) → PROCESS (clean/fact-sentence/chunk)
      → EMBED (BGE-small-v1.5) → FAISS (cosine) → RETRIEVE (+muni filter)
      → GROUNDING PROMPT → Qwen2.5-3B-Instruct → CITED ANSWER
      → EVAL (recall@k · faithfulness · numeric-support · abstention)
```

## Real data sources (all free, no auth)
- **Wikipedia** (Action API `prop=extracts`) — descriptive text per municipality.
- **Wikivoyage** (same MediaWiki API) — editorial, qualitative "what's it like" prose;
  labelled as travel-writer tone, not resident 

In [29]:
#@title List deliverables on disk
import os
print("Deliverables written:\n")
for base in ["README.md", "requirements.txt"]:
    if os.path.exists(base):
        print(f"  {base:32s} {os.path.getsize(base):>10,} bytes")
for f in sorted(Path(CONFIG.artifacts_dir).glob("*")):
    print(f"  {str(f):32s} {f.stat().st_size:>10,} bytes")
for f in sorted(Path(CONFIG.raw_dir).glob("*")):
    print(f"  {str(f):32s} {f.stat().st_size:>10,} bytes")

Deliverables written:

  README.md                             5,053 bytes
  requirements.txt                        308 bytes
  ontario_rag_data/artifacts/README.md      5,053 bytes
  ontario_rag_data/artifacts/chunks.json    609,233 bytes
  ontario_rag_data/artifacts/eval_report.md      1,082 bytes
  ontario_rag_data/artifacts/eval_results.csv      7,558 bytes
  ontario_rag_data/artifacts/faiss.index    872,493 bytes
  ontario_rag_data/artifacts/requirements.txt        308 bytes
  ontario_rag_data/raw/corpus.json  1,772,343 bytes


## 12 · Limitations & honest self-assessment

**Scope deliberately cut to guarantee a clean free-Colab run:**
- **StatCan WDS is wired but off by default.** Matching full census tables to
  municipalities reliably is fragile; population/density come from the cited
  2021-Census backbone instead. Turn on `CONFIG.use_statcan_wds` to fetch live.
- **Ontario Open Data (CKAN)** and a **cross-encoder reranker** are described but not
  in the default path, to keep memory and latency within free-tier limits.
- **Wikipedia text is capped** per article (intro + early sections).

**Inherent limitations (cannot be fully engineered away here):**
- Dense retrieval has no notion of truth — it retrieves *similar* text, which can be
  outdated or context-poor.
- A small local LLM can still hallucinate or misattribute despite the grounding
  prompt; the numeric-support check catches fabricated numbers but not every
  misstatement. Note the check verifies a number appears in *some* retrieved chunk,
  **not** that it is the most current/authoritative one — a comparison could quote a
  stale descriptive population if the census card weren't retrieved, which is why
  comparison queries now force-include each city's 2021-Census fact card.
- OSM/Census coverage is uneven across municipalities and boundary definitions differ
  between sources.

**What the evaluation does and doesn't prove:** it shows the retriever surfaces the
right *source type* for known questions and that the system abstains on out-of-scope
queries — it does **not** certify every generated sentence is correct. Treat outputs
as sourced pointers to verify, not as authoritative advice on where to live.

**This assistant reports what public data says. It does not tell anyone where to move.**